# 06 Reconcile and report coverage
Compare each filing's cleaned holdings with **the filer's own reported total**. Admit filings within the tolerance, count **every** filing you attempted, report coverage honestly, and gather evidence on every filing you could not admit.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
if not (ROOT / ".notebook-scratch").exists():
    raise RuntimeError("This teaching notebook writes outputs. Run tools/run_notebooks.py in its isolated scratch copy; use dvc repro for submission outputs.")
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
sub, tag = load.load_sub(RDIR), load.load_tag(RDIR)
header = load.soi_header(RDIR)
colmap = mapping.map_soi_columns(header, tag)
soi = load.load_soi(RDIR, usecols=mapping.soi_usecols(header, colmap))   # only the columns we use
soi = soi[soi["adsh"].isin(sub["adsh"])]                                  # drop superseded filings
canon = holdings.split_periods(mapping.to_canonical(soi, colmap), sub)  # current period from SUB
num = load.load_num(RDIR, tags=[reconcile.FV_TAG], custom_fv=True)   # plus custom-tag fair values, for diagnosis
h = holdings.coalesce_holdings(canon)
kept, log = clean.apply_rules(h[h["is_current"]], [clean.rule_totals, clean.rule_cash])
filed = reconcile.filed_totals(num, sub)
rec = reconcile.reconcile(kept, filed, sub, P["reconcile"]["tolerance"], soi_adsh=set(soi["adsh"]))
rec[["name", "n_holdings", "holdings_fv", "filed_fv", "gap_pct", "admitted"]]

In [ ]:
cov = reconcile.coverage(rec, sub)
print(f"Admitted {cov['filings_admitted']} of {cov['filings_attempted']} filings ({cov['share_of_filings']:.0%}), "
      f"covering {cov['share_of_dollars']:.0%} of filed fair value")
print("Current filings in SUB with no SOI rows at all (report them separately):", cov["filings_without_soi_rows"])

In [ ]:
diag = reconcile.diagnose(rec, kept, num, canon)
diag

In [ ]:
paths_out = report.write_outputs(kept, rec, diag, log, ROOT, P)
for k, v in paths_out.items():
    print(f"  {k:9s} -> {v.relative_to(ROOT)}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.hist(rec["gap_pct"].dropna().clip(-0.5, 0.5) * 100, bins=40, color="#13837A")
ax.axvspan(-P["reconcile"]["tolerance"] * 100, P["reconcile"]["tolerance"] * 100, color="#E8A33D", alpha=0.3, label="tolerance")
ax.set_xlabel("gap (%), clipped at ±50"); ax.set_ylabel("filings"); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / P["outputs"]["reports"] / "gap_histogram.png", dpi=150); plt.show()

**Your turn.** On your release, coverage under these two rules will be partial. Raise it honestly: each new rule must be motivated by an artifact and tested, and every filing you still cannot admit gets a root-cause category and an example in `reports/triage.md`. Then run the self-check from your `cs2b/` folder: `PYTHONPATH=src python -m lookthrough.check`.

The triage-category check **fails until you triage**: `report.write_outputs` marks every excluded filing `to_triage`. Replace each with one of the categories in Case Study 2B, Section 7, once you have the evidence.

In [ ]:
from lookthrough import check
for name, ok, detail in check.run_checks(ROOT):
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"  [{detail}]" if not ok and detail else ""))